In [1]:
import logging
from pathlib import Path
import re
import subprocess

import pandas as pd

In [2]:
logger = logging.getLogger()

In [79]:
# Custom words to remove
EXCLUDE_LIST = ['[FREE DL]', '[DUPLOC10YRS]', '[Digital Bonus]', '[Premiere]', '[Bassrush Premiere]', '(DDD118)']
REPLACE_DICT = {
    "¿": "?"
}

In [4]:
! ls /c/Users/sesti/Music/Playlists/djutil/stage/

bass
pop


In [5]:
! ls /c/Users/sesti/Music/Playlists/djutil/stage/*

/c/Users/sesti/Music/Playlists/djutil/stage/bass:
1.CESCO - Flump.mp3
1.Criso - Rinse Out.mp3
1.Ferrofluid - Convex.mp3
1.Jah Beer - Castle.mp3
1.NINA - Ethereal.mp3
1.Teffa - Toastin.mp3
1111 x BIIANCO - Against The Wall.mp3
12.Ternion Sound - OnPark (Criso Remix).mp3
19.VISION - Gowchii - Rewind The Tune.mp3
2.DÃŠTRE - Fuck It Up.mp3
2.gyrofield - Banned Fiction.mp3
2.NINA - Trapped Nerve.mp3
2.Sub Basics - Shipment Pt.2.mp3
3.Criso & Joe Nice - Knowledge.mp3
3.Monuman - Inflict.mp3
4.ALIX PEREZ - Skunkworks ft. SPMC.mp3
4.Criso - Bout This.mp3
4.Throwing Snow - Forged.mp3
5.Nina - Backshot (Dub) [Digital Bonus].mp3
6.Alix Perez - Poison Ivy.mp3
6.Kercha - Street (Shanti. Re;Funk).mp3
Canmore Ceremony (Orenda Remix)_2248777682 - Orenda.m4a
Fake Two Piece x Floret Loret - Ain't no time for hesitation_2054381408 - Fake Two Piece.m4a
Feel_1875756174 - 1111.m4a
Ferrofluid - Jealousy_1889891832 - Qualia.m4a
Ferrofluid - Sobel.m4a
Hamdi - Skanka VIP.mp3
Hijinx - Swarm (I.D3 Bootleg) [FREE 

In [6]:
def get_file_df(dir_path, glob_pat="*"):
    path = []
    parent = []
    name = []
    stem = []
    suffix = []
    for pp in dir_path.glob(glob_pat):
        path.append(pp.resolve())
        parent.append(pp.parent)
        name.append(pp.name)
        stem.append(pp.stem)
        suffix.append(pp.suffix)
    df = pd.DataFrame({"path": path, "parent": parent, "name": name, "stem": stem, "suffix": suffix})
    return df

In [25]:
def strip_digit_prefix(stem):
    # "2.Sub Basics - Shipment Pt.2" => "Sub Basics - Shipment Pt.2"
    return re.sub(r'^\d+\.', '', stem)

In [8]:
def strip_underscore_digits(stem):
    # "Criso & Breathworkz - Wicked_2051092604 - Criso" => "Criso & Breathworkz - Wicked - Criso"
    return re.sub(r'_\d+', '', stem)

In [87]:
def remove_words(series):
    replace = REPLACE_DICT
    remove = {s: "" for s in EXCLUDE_LIST}
    replace.update(remove)
    series = series.str.replace(replace)
    # Remove consecutive whitespace and strip
    series = series.str.replace(r'\s+', ' ', regex=True).str.strip()
    return series

# Action

In [88]:
src_dir = Path("C:/Users/sesti/Music/Playlists/djutil/stage/bass")
dst_dir = Path("C:/Users/sesti/Music/Playlists/bass")

In [89]:
df = get_file_df(src_dir)
df["new_stem"] = df.stem.map(strip_digit_prefix).map(strip_underscore_digits)
df["new_stem"] = remove_words(df.new_stem).str.strip()
df["new_suffix"] = ".wav"
df["new_parent"] = dst_dir
df["new_sample_rate"] = str(44100)
df["new_sample_fmt"] = "s16"
df["new_path"] = df.apply(lambda row: row.new_parent / (row.new_stem + row.new_suffix), axis=1).values


In [93]:
df.new_stem.head()

0          CESCO - Flump
1      Criso - Rinse Out
2    Ferrofluid - Convex
3      Jah Beer - Castle
4        NINA - Ethereal
Name: new_stem, dtype: str

In [94]:
df.new_path.head()

0    C:\Users\sesti\Music\Playlists\bass\CESCO - Fl...
1    C:\Users\sesti\Music\Playlists\bass\Criso - Ri...
2    C:\Users\sesti\Music\Playlists\bass\Ferrofluid...
3    C:\Users\sesti\Music\Playlists\bass\Jah Beer -...
4    C:\Users\sesti\Music\Playlists\bass\NINA - Eth...
Name: new_path, dtype: object

In [95]:
def run_ffmpeg(row):
    assert row.path
    assert row.new_sample_fmt
    assert row.new_sample_rate
    assert row.new_path
    if row.new_path.is_file():
        logger.warning(f"File exists, skipping: {row.new_path}")
        return None
    else:
        command = [
            'ffmpeg',
            '-i', row.path,
            '-sample_fmt', row.new_sample_fmt,
            '-ar', row.new_sample_rate,
            row.new_path
        ]
        process = subprocess.Popen(command)
        return_code = process.wait()
        return int(return_code)
        
    

In [96]:
df.apply(run_ffmpeg, axis=1)

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    0
20    0
21    0
22    0
23    0
24    0
25    0
26    0
27    0
28    0
29    0
30    0
31    0
32    0
33    0
34    0
35    0
36    0
37    0
38    0
39    0
40    0
41    0
42    0
43    0
44    0
45    0
46    0
47    0
48    0
49    0
50    1
51    0
52    0
53    0
54    0
55    0
dtype: int64

In [97]:
! ls $dst_dir

1111 x BIIANCO - Against The Wall.wav
Alix Perez - Poison Ivy.wav
ALIX PEREZ - Skunkworks ft. SPMC.wav
Canmore Ceremony (Orenda Remix) - Orenda.wav
CESCO - Flump.wav
Criso - Bout This.wav
Criso - Rinse Out.wav
Criso & Joe Nice - Knowledge.wav
DÃŠTRE - Fuck It Up.wav
Fake Two Piece x Floret Loret - Ain't no time for hesitation - Fake Two Piece.wav
Feel - 1111.wav
Ferrofluid - Convex.wav
Ferrofluid - Jealousy - Qualia.wav
Ferrofluid - Sobel.wav
gyrofield - Banned Fiction.wav
Hamdi - Skanka VIP.wav
Hijinx - Swarm (I.D3 Bootleg).wav
Hijinx - Swarm.wav
Jah Beer - Castle.wav
jordnmoody, Aztek - BADAC!D.wav
Kercha - Feature - Kercha.wav
Kercha - Street (Shanti. Re;Funk).wav
Kercha, Ghostek - Ghetto Jazz - Ghostek Â¦ Mxstakes.wav
Killashi - Jadon Woods.wav
Klinical - Answer.wav
Klinical - Made It Clear - Overview Music.wav
LabRat - Slapz.wav
Lowcation x Chmura x Phurn - Realms - Qualia.wav
Medicine Place - Hieroglyph - Wubaholics.wav
Monuman - Inflict.wav
Mr. Bill & bandoum - Lotus.wav
Mr. Bil